# Skill catalog review

สร้าง preview พร้อมเทียบชื่อกับ catalog ปัจจุบันใน DB แบบอ่านอย่างเดียว:
```powershell
python -m scripts.preview_skill_catalog_rebuild --suggest-limit 100 --resume --compare-database-catalog
```

รีวิวแบบถามทีละ candidate และบันทึกเฉพาะ CSV (ไม่แก้ DB):
```powershell
python -m scripts.review_skill_catalog_preview
```

In [18]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
CSV_PATH = PROJECT_ROOT / 'data' / 'processed' / 'skill_candidates' / 'catalog_rebuild_preview.csv'
if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Run the preview command first: {CSV_PATH}')

df = pd.read_csv(CSV_PATH, encoding='utf-8-sig').fillna('')
for column in ['observed_names', 'source_runs', 'career_names', 'example_job_titles', 'example_evidence']:
    if column in df:
        df[column] = df[column].apply(lambda value: json.loads(value) if isinstance(value, str) and value else [])
print(f'{len(df)} candidate groups from {CSV_PATH}')
print(f"Reviewed: {(df.review_decision != '').sum()} | Needs suggestion: {(df.preview_status == 'needs_llm_suggestion').sum()} | Manual review: {(df.preview_status == 'manual_review').sum()} | Ready: {(df.preview_status == 'ready_for_approval').sum()}")

700 candidate groups from c:\Users\PC\Workspace\competition\ntt-data\glow-job-data-pipeline\data\processed\skill_candidates\catalog_rebuild_preview.csv
Reviewed: 0 | Needs suggestion: 600 | Manual review: 36 | Ready: 64


## Candidate เทียบกับชื่อที่มีอยู่

แก้ `SEARCH` เพื่อค้นชื่อ เช่น `AWS` แล้วดูคอลัมน์เทียบกันในแถวเดียว: bootstrap, exact canonical/alias ใน DB, ชื่อ DB ที่ใกล้เคียง, และชื่อ candidate ที่อาจซ้ำกันใน preview. Similarity เป็นตัวช่วยค้นเท่านั้น ไม่ได้ยืนยันว่าเป็น skill เดียวกัน. ถ้า column DB ว่าง ให้รัน preview ด้วย `--compare-database-catalog`.

In [19]:
SEARCH = 'AWS'  # เปลี่ยนเป็นชื่อ candidate ที่ต้องการดู
mask = df['candidate_name'].astype(str).str.contains(SEARCH, case=False, na=False)
compare_columns = [
    'candidate_name', 'observed_names', 'job_count', 'occurrence_count',
    'possible_existing_skill', 'existing_database_matches', 'similar_database_skills',
    'possible_duplicate_candidates', 'suggestion_decision', 'suggested_skill_name',
    'suggested_skill_category', 'confidence_level', 'preview_status',
    'review_decision', 'canonical_skill_name', 'skill_category',
]
display(df.loc[mask, [column for column in compare_columns if column in df.columns]].reset_index(drop=True))

,candidate_name,observed_names,job_count,occurrence_count,possible_existing_skill,possible_duplicate_candidates,suggestion_decision,suggested_skill_name,suggested_skill_category,confidence_level,preview_status,review_decision,canonical_skill_name,skill_category
0,AWS,[AWS],16,16,,,new_skill,AWS,other,high,ready_for_approval,,,
1,AWS DynamoDB,[AWS DynamoDB],2,2,,,,,,,needs_llm_suggestion,,,
2,"AWS Bedrock Agents, Amazon Q, or Google Vertex...","[AWS Bedrock Agents, Amazon Q, or Google Verte...",1,1,,,,,,,needs_llm_suggestion,,,
3,AWS serverless infrastructure,[AWS serverless infrastructure],1,1,,,,,,,needs_llm_suggestion,,,
4,"cloud data platforms (AWS, Azure, GCP)","[cloud data platforms (AWS, Azure, GCP)]",1,1,,,,,,,needs_llm_suggestion,,,


## รายการที่ควรเปิดรีวิว

ตารางนี้แสดงรายการที่ยังไม่มี manual decision และยังไม่พร้อมอนุมัติอัตโนมัติ. หลักฐาน posting และผลเทียบ DB แสดงในแถวเดียวกัน. ใช้ command `review_skill_catalog_preview` เพื่อบันทึก decision ลง CSV; คำสั่งนั้นอ่าน DB catalog เท่านั้น.

In [20]:
pending = df[(df['review_decision'] == '') & (df['preview_status'] == 'manual_review')].copy()
review_columns = [
    'candidate_name', 'observed_names', 'job_count', 'possible_existing_skill',
    'existing_database_matches', 'similar_database_skills', 'possible_duplicate_candidates',
    'suggested_skill_name', 'suggested_skill_category', 'confidence_level',
    'suggestion_reason', 'example_job_titles', 'example_evidence',
]
display(pending[[column for column in review_columns if column in pending.columns]].reset_index(drop=True))

,candidate_name,observed_names,job_count,possible_existing_skill,possible_duplicate_candidates,suggested_skill_name,suggested_skill_category,confidence_level,suggestion_reason,example_job_titles,example_evidence
0,CI/CD pipelines,[CI/CD pipelines],11,,CI/CD; CI/CD Pipeline; CI/CD tools,CI/CD pipelines,other,high,CI/CD pipelines represent a software developme...,"[Data Engineer Specialist, Senior Backend Engi...",[• Experience with version control (Git) and C...
1,CI/CD,[CI/CD],9,,CI/CD Pipeline; CI/CD pipelines; CI/CD tools,,,high,"CI/CD is a software engineering practice, not ...","[BI Engineer, AI Software Engineer - Assessmen...","[Knowledge of version control tools (e.g., Git..."
2,AI,[AI],8,,,,,high,AI is a broad field of study and general conce...,"[Data Analyst (Fulltime & Freelance), นักพัฒนา...",[• ประสบการณ์ในการทำ Machine Learning และ AI จ...
3,data engineering,"[data engineering, Data Engineering]",8,,data engineering tools,data engineering,other,high,Distinct professional practice related to buil...,"[Senior Data Engineer I (Finance), Global Data...","[5-8 years in data engineering, including buil..."
4,data modelling,[data modelling],7,,data modeling,data modelling,data_practice,high,Distinct data practice for structuring and des...,"[Data Engineer Specialist, Global Data Platfor...","[• Proficiency in SQL and data modelling., Str..."
5,Go,[Go],7,,Golang,Go,programming_language,high,Well-known programming language distinctly evi...,"[Senior Software Engineer, Data Infrastructure...",[Production automation experience in Python or...
6,Node.js,[Node.js],7,,Nodejs,Node.js,programming_language,high,Node.js is a distinct programming platform/lan...,"[Software Engineer - WFH 95% + Office 5%, Back...","[พัฒนา Microservice ด้วย Node.js, GraphQL และอ..."
7,BI tools,[BI tools],6,,,,,high,BI tools is a broad category description rathe...,"[BI Engineer, Senior Specialist – Data Scienti...",[Proficient in SQL and one or more BI tools (e...
8,data analysis,"[data analysis, Data analysis]",6,,,,,high,data analysis is a general field/task descript...,"[Data Scientist Intern, Data Quality Assurance...","[A good foundation in Python, data analysis, a..."
9,data modeling,[data modeling],6,,data modeling techniques; data modelling,data modeling,other,high,Distinct skill related to data structure desig...,"[Senior Data Engineer I (Finance), BI Engineer...","[Experience with data modeling, dimensional de..."
